Salary Scheduling:
- Add golden datasets for UC and CSU to ./golden-datasets
- Use salary logic to add salary information to golden CSU dataset (from ../DataPrep/csu_data_prep.ipynb) and data is in SalaryRecsCSU.csv (download from https://www.calstate.edu/csu-system/careers/compensation/Pages/salary-schedule.aspx)
- Clean CSU class A to "lecturer"


Joined Dataset:
- Include columns for low, high and mean salary schedule (mean would be the expected)
- include scraped title, transparent california title, and assigned salary schedule “rank” 
- Rename rank A to lecturer A (I think maybe we should denote A-D in lecturer assignment)



In [36]:
import pandas as pd

salary_schedule_df = pd.read_csv('./golden-datasets/SalaryRecsCSU.csv')
csu_data_df = pd.read_parquet('./golden-datasets/golden_csu_2013-2024.parquet')
uc_data_df = pd.read_parquet('./golden-datasets/golden_uc_2013-2024.parquet')

In [37]:
csu_data_df['campus'].unique()

<ArrowStringArray>
[                             'CSU Bakersfield',
                          'CSU Channel Islands',
                               'CSU Long Beach',
                              'CSU Los Angeles',
                               'CSU Northridge',
                           'CSU San Bernardino',
                            'Cal Poly Humboldt',
                              'Cal Poly Pomona',
                     'Cal Poly San Luis Obispo',
       'California State University San Marcos',
           'California State University, Chico',
 'California State University, Dominguez Hills',
        'California State University, East Bay',
          'California State University, Fresno',
       'California State University, Fullerton',
    'California State University, Monterey Bay',
      'California State University, Sacramento',
      'California State University, Stanislaus',
                   'San Diego State University',
               'San Francisco State University',
 

### Match CSU roles and titles to the 2024 schedule

Use only the golden CSU dataset and `SalaryRecsCSU.csv`. Apply the same `CSU_ROLES`
rules used by the pipeline to the schedule's `Class Title`, then match against `role`.
All payroll years use the December 31, 2024 schedule cutoff.

For the main Instructional Faculty (`ladder_faculty`) and Lecturer (`lecturer`)
classifications, combine academic-year (AY) and 12-month schedules **within each rank/range**:

- `Min Salary` = the lower of the AY and 12-month minimums.
- `Max Salary` = the higher of the AY and 12-month maximums.
- These are monthly bounds (`Salary Per = M`); both schedules have `Annual Factor = 12`.

This produces an encompassing comparison range because `role` does not retain appointment
length. It is not an individual's actual appointment-specific range. Professor ranks and
lecturer ranges A-D remain separate. For example, Lecturer A spans $5,507-$7,658 per month
using the supplied 2024 rates.

Combined rows are labeled `AY / 12 MONTH` and list both source class codes. Their effective
date is the latest contributing effective date. Other metadata is retained only when both
source rows agree; differing metadata is left missing. `Minimum Rate` and `Maximum Rate`
also use the combined bounds.

Keep the reference notebook's title/rank parsing, lecturer-A and Instructor overrides,
and unresolved-match handling. Those overrides also use the combined AY/12-month ranges.
When several classifications fit, use the lowest class code among usable monthly candidates
only if their rank, minimum, maximum, and annual factor agree. This resolves Special Programs
versus Extension to class 2422 with the shared monthly range; nonnumeric/unit-based rows are
not used for this default. The match method marks the assumption. This chooses a comparison
schedule, not the person's verified payroll classification. Turn off
`ASSIGN_EQUIVALENT_MONTHLY_RANGES` to keep these cases unmatched. Differing ranges stay unresolved.

The result is `final_csu_df`, with range A labeled `lecturer class A`.

### Adjustable assumptions and function layout

Edit the constants in the next cell, then rerun the following cells in order.
The functions are grouped into title/role parsing, schedule preparation, AY/12-month
combination, candidate selection, matching, and joining. The last cell runs those steps.

`CSU_ROLES` controls the ordered groupings; `FACULTY_CLASSIFICATIONS` selects the main
faculty classes; `BOUND_AGGREGATIONS` controls the enclosing minimum/maximum;
`LECTURER_DEFAULT_RANGE` controls the lecturer default. Explicit title overrides run
before the role default, so exact `Instructional Faculty` still selects Instructor.
Comments beside each setting describe the other assumptions and ambiguity rules.

In [38]:
import re
from functools import lru_cache

# ASSUMPTION: compare every payroll year against this one schedule year.
# Change this year to change the cutoff; this does not perform a historical join.
SALARY_SCHEDULE_YEAR = 2024
SALARY_CUTOFF = pd.Timestamp(SALARY_SCHEDULE_YEAR, 12, 31)

# ASSUMPTION: use the data-pipeline's role groupings exactly, in this order.
# First match wins: summer/extension instructional faculty must be supplemental,
# and post-doctoral counselor interns must be postdocs rather than trainees.
#these are the roles used to build the CSU golden dataset in ../data-pipeline so these same rules
# are applied to the salary schedule csv to classify them
CSU_ROLES = [
    ("supplemental_teaching_pay", r"special programs|summer session|extension|additional employment|music studio"),
    ("ladder_faculty", r"^instructional faculty|^grant-related/specially funded instructional faculty|^department chair|^cma instructional faculty"),
    ("lecturer", r"lecturer"),
    ("clinical_adjunct_faculty", r"^visiting faculty"),
    ("postdoc", r"^research fellow|post-doctoral"),
    ("trainee", r"counselor intern"),
    ("researcher_specialist_librarian", r"librarian|academic-related|coach"),
]

# ASSUMPTION: these broad roles use the main faculty classifications.
# Maritime, grant-funded, etc. are not separately inferred from a broad role.
# Other roles keep all their candidate classes and may remain ambiguous.
FACULTY_CLASSIFICATIONS = {
    "ladder_faculty": "instructional faculty",
    "lecturer": "lecturer",
}

# ASSUMPTION: combine only AY and 12-month schedules, within the same rank.
# Keep the lowest minimum and highest maximum, not an average of the schedules.
# Both source rows must exist and use the same monthly/annual-factor convention.
APPOINTMENT_TERMS = ("academic year", "12 month")
COMBINED_TERM_LABEL = "AY / 12 MONTH"
BOUND_AGGREGATIONS = {"Min Salary": "min", "Max Salary": "max"}
EXPECTED_SALARY_UNIT = "M"
EXPECTED_ANNUAL_FACTOR = 12

# ASSUMPTION: lecturer titles and lecturer roles default to A, even if the
# scraped title says Lecturer B/C/D or I/II/III/IV. Change this one value
# to use a different default; honoring explicit levels needs a different rule.
LECTURER_DEFAULT_RANGE = "a"

# ASSUMPTION: explicit title rules take precedence over the role default.
# Exact "Instructional Faculty" selects Instructor; lecturer titles select A.
# If the role lacks that range, try the main combined faculty classification.
TITLE_OVERRIDE_CLASSIFICATIONS = {
    "instructional faculty": "instructional faculty",
    "lecturer": "lecturer",
}

# ASSUMPTION: unranked chair/emeritus titles imply Professor. Explicit
# Assistant/Associate Professor ranks remain distinct. Long biographies and
# conflicting ranks remain unresolved instead of choosing an arbitrary rank.
MAX_TITLE_LENGTH = 350
UNRANKED_CHAIR_EMERITUS_RANK = "professor"

# Reference compatibility rules for substitute-faculty range labels.
COMBINED_RANK_LABELS = {
    "a asst instructor": {"instructor"},
    "b asst professor": {"assistant professor"},
    "c assoc prof prof": {"associate professor", "professor"},
}

# Presentation only: this does not change the numeric schedule range code.
FINAL_RANGE_LABELS = {"a": "lecturer class A", "range a": "lecturer class A"}

# ASSUMPTION: if several rows fit, prefer usable monthly ranges when their
# rank, bounds, pay unit, and annual factor are identical. Choose the lowest
# class code deterministically. This is an assumed comparison schedule, not
# recovery of the actual payroll classification (which could use another unit).
ASSIGN_EQUIVALENT_MONTHLY_RANGES = True

# ASSUMPTION: these roles are never compared to the salary schedule. Any row whose role is in this
# set is left unmatched (no schedule columns attached) and reported under "excluded_role_records"
# in the summary tables. supplemental_teaching_pay is extra pay on top of a main appointment,
# so an annual salary range does not apply to it. Add other role names here to exclude them too.
EXCLUDED_COMPARISON_ROLES = {"supplemental_teaching_pay"}
# The text written to `unmatched_reason` for excluded rows. The summaries use this same constant
# to tell "excluded on purpose" apart from "could not be matched".
EXCLUDED_ROLE_REASON = "Role excluded from salary comparison"

# ASSUMPTION: the title "Instructional Faculty" by itself is the generic CSU class name, not a rank,
# so it does not mean Instructor. Options:
#   "instructor"             -> old behavior: treat it as the Instructor range
#   "unmatched"              -> leave the rank unknown (row stays unmatched)
#   "infer_from_other_years" -> look at the same person (same campus + name) in their other years;
#                               use the rank from the closest year whose title states one
#                               (ties -> earlier year; skipped if the closest years disagree).
#                               If there is none, the row stays unmatched.
INSTRUCTIONAL_FACULTY_TITLE_POLICY = "infer_from_other_years"
INFERABLE_RANKS = {"instructor", "assistant professor", "associate professor", "professor"}

# Summary tables only: restrict them to one payroll year (None = all years).
SUMMARY_YEAR = 2024

# A record is only compared to the schedule when its pay is strictly greater than this.
# 0 means zero-pay records are counted separately (zero_pay_records) and left out of the percentages.
MIN_PAY_FOR_COMPARISON = 0


In [39]:
# Text normalization and rank extraction follow the reference notebook.
RANK_PATTERN = re.compile(
    r"\b(?:(assistant|asst|assis|associate|associat|assoc|assc)\s+"
    r"(?:(?:clinical|research|visiting|adjunct)\s+)*)?"
    r"(professor|prof)\b"
    r"|\b(lecturer|lecturtrer|instructor)(?:\s+(iii|ii|iv|i|[a-d]))?\b"
)


def normalize_title(value):
    """Normalize scraped titles, including joined words such as CoordinatorAssociate."""
    if pd.isna(value):
        return ""
    separated = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", str(value))
    return re.sub(r"[^a-z0-9]+", " ", separated.casefold()).strip()


def normalize_class_titles(titles):
    """Normalize schedule classification names without removing hyphens."""
    return (
        titles.astype("string").str.strip().str.casefold()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"\s*-\s*", "-", regex=True)
        .replace("", pd.NA)
    )


def classify_schedule_role(class_title):
    """Apply the ordered CSU_ROLES rules to a schedule class title."""
    text = "" if pd.isna(class_title) else str(class_title).lower().strip()
    for role, pattern in CSU_ROLES:
        if re.search(pattern, text):
            return role
    return "staff"


def rank_from_title_match(match):
    """Translate one regex match into a distinct professor or lecturer rank."""
    qualifier, professor, role, level = match.groups()
    if professor:
        if qualifier in {"assistant", "asst", "assis"}:
            return "assistant professor"
        return "associate professor" if qualifier else "professor"
    role = "lecturer" if role == "lecturtrer" else role
    return role + (f" {level}" if level else "")


def is_generic_instructional_faculty(title):
    """True when the whole title is just the generic class name 'Instructional Faculty'."""
    return normalize_title(title) == "instructional faculty"


def extract_title_rank(title):
    """Return a single rank; leave long or conflicting titles unresolved."""
    text = normalize_title(title)
    if len(text) > MAX_TITLE_LENGTH:
        return pd.NA
    ranks = {rank_from_title_match(match) for match in RANK_PATTERN.finditer(text)}
    if not ranks:
        if re.search(r"\b(?:emerit(?:us|a|i|u)?|chair(?:person|man|woman)?)\b", text):
            return UNRANKED_CHAIR_EMERITUS_RANK
        if text == "instructional faculty" and INSTRUCTIONAL_FACULTY_TITLE_POLICY == "instructor":
            return "instructor"
    return next(iter(ranks)) if len(ranks) == 1 else pd.NA


def determine_title_rule(title, rank):
    """Identify only the explicit title overrides supported by the reference."""
    if is_generic_instructional_faculty(title) and INSTRUCTIONAL_FACULTY_TITLE_POLICY == "instructor":
        return "instructional faculty"
    if pd.notna(rank) and rank.startswith("lecturer"):
        return "lecturer"
    return None

In [40]:
def parse_salary_amounts(values):
    """Convert dollar strings to numeric amounts; invalid values remain missing."""
    return pd.to_numeric(
        values.astype("string").str.replace(r"[$,]", "", regex=True), errors="coerce"
    )


def filter_schedule_to_cutoff(raw_schedule, cutoff):
    """Keep the latest eligible version of each class/range without changing the input."""
    schedule = raw_schedule.copy()
    schedule["Effective Date"] = pd.to_datetime(
        schedule["Effective Date"], format="%m/%d/%Y", errors="coerce"
    )
    schedule = schedule.loc[schedule["Effective Date"].le(cutoff)].copy()
    latest_dates = schedule.groupby(["Class", "Range"], dropna=False)[
        "Effective Date"
    ].transform("max")
    # Exact duplicate rows are harmless; conflicting rows stay visible so that
    # matching reports ambiguity instead of silently taking the first one.
    return schedule.loc[schedule["Effective Date"].eq(latest_dates)].drop_duplicates().reset_index(drop=True)


def prepare_schedule_columns(schedule):
    """Add normalized matching keys and parse salary bounds."""
    schedule = schedule.copy()
    schedule["_salary_row_id"] = pd.array(range(len(schedule)), dtype="Int64")
    schedule["_role_key"] = schedule["Class Title"].map(classify_schedule_role)
    schedule["_class_key"] = normalize_class_titles(schedule["Class Title"])
    schedule["_range_key"] = schedule["Range Title"].map(normalize_title)
    for column in BOUND_AGGREGATIONS:
        schedule[column] = parse_salary_amounts(schedule[column])
    schedule["Class"] = schedule["Class"].astype("string")
    return schedule

In [41]:
def combined_class_key(classification):
    """Key used by both the combined schedule rows and matching defaults."""
    return classification + "-" + COMBINED_TERM_LABEL.casefold()


def validate_appointment_pair(rows, expected_class_keys):
    """Require one compatible source row per appointment term for this rank."""
    assert len(rows) == len(expected_class_keys), "Appointment pair is incomplete or ambiguous"
    assert set(rows["_class_key"]) == expected_class_keys, "Appointment terms do not match"
    assert rows["Salary Per"].eq(EXPECTED_SALARY_UNIT).all(), "Salary units differ"
    assert rows["Annual Factor"].eq(EXPECTED_ANNUAL_FACTOR).all(), "Annual factors differ"
    assert rows[list(BOUND_AGGREGATIONS)].notna().all().all(), "Salary bounds are missing"


def combine_one_rank(rows, classification):
    """Build an encompassing AY/12-month row without mixing different ranks."""
    # ASSUMPTION: preserve metadata only when both source rows agree.
    # A differing SSI rate, footnote, etc. is missing in the combined row.
    combined = {
        column: rows[column].iloc[0] if rows[column].nunique(dropna=False) == 1 else pd.NA
        for column in rows.columns
    }
    for column, aggregation in BOUND_AGGREGATIONS.items():
        combined[column] = rows[column].agg(aggregation)
    combined.update({
        "Class": " | ".join(sorted(rows["Class"].unique())),
        "Class Title": classification.upper() + " - " + COMBINED_TERM_LABEL,
        "_class_key": combined_class_key(classification),
        # The combined row is valid only once both source rates are effective.
        "Effective Date": rows["Effective Date"].max(),
        "Minimum Rate": combined["Min Salary"],
        "Maximum Rate": combined["Max Salary"],
    })
    return combined


def combine_appointment_ranges(schedule):
    """Replace the configured AY/12-month pairs; leave other classifications intact."""
    combined_rows = []
    replaced_row_ids = []
    for classification in dict.fromkeys(FACULTY_CLASSIFICATIONS.values()):
        class_keys = {classification + "-" + term for term in APPOINTMENT_TERMS}
        appointment_rows = schedule.loc[schedule["_class_key"].isin(class_keys)]
        for _, rank_rows in appointment_rows.groupby("_range_key", dropna=False):
            validate_appointment_pair(rank_rows, class_keys)
            combined_rows.append(combine_one_rank(rank_rows, classification))
            replaced_row_ids.extend(rank_rows["_salary_row_id"].tolist())
    unchanged_rows = schedule.loc[~schedule["_salary_row_id"].isin(replaced_row_ids)]
    combined_schedule = pd.concat([unchanged_rows, pd.DataFrame(combined_rows)], ignore_index=True)
    combined_schedule["_salary_row_id"] = pd.array(range(len(combined_schedule)), dtype="Int64")
    return combined_schedule


def build_matching_schedule(raw_schedule, cutoff):
    """Prepare the cutoff-specific schedule and combine the configured appointments."""
    eligible = filter_schedule_to_cutoff(raw_schedule, cutoff)
    prepared = prepare_schedule_columns(eligible)
    return combine_appointment_ranges(prepared)

In [42]:
def lecturer_range_keys():
    """Accept both spellings of the configured default lecturer range."""
    return {LECTURER_DEFAULT_RANGE, "range " + LECTURER_DEFAULT_RANGE}


def select_role_candidates(schedule, groups_by_role, role):
    """Restrict main faculty roles to their combined classification."""
    candidates = groups_by_role.get(role, schedule.iloc[:0])
    if role in FACULTY_CLASSIFICATIONS:
        class_key = combined_class_key(FACULTY_CLASSIFICATIONS[role])
        candidates = candidates.loc[candidates["_class_key"].eq(class_key)]
    return candidates


def apply_title_override(schedule, candidates, title_rule):
    """Prefer the requested range within the role, then try the main faculty class."""
    if title_rule not in TITLE_OVERRIDE_CLASSIFICATIONS:
        return candidates, None
    instructor_rule = title_rule == "instructional faculty"
    preferred_ranges = {"instructor"} if instructor_rule else lecturer_range_keys()
    within_role = candidates.loc[candidates["_range_key"].isin(preferred_ranges)]
    if not within_role.empty:
        label = "Instructor" if instructor_rule else "Range " + LECTURER_DEFAULT_RANGE.upper()
        return within_role, "title rule: " + label
    classification = TITLE_OVERRIDE_CLASSIFICATIONS[title_rule]
    fallback = schedule.loc[
        schedule["_class_key"].eq(combined_class_key(classification))
        & schedule["_range_key"].isin(preferred_ranges)
    ]
    return fallback, "title override: " + classification + " / AY and 12 month"


def describe_available_ranges(candidates):
    """Keep the available labels in diagnostics when no unique match can be made."""
    labels = candidates["Range Title"].fillna("(blank)").astype(str).str.strip()
    return " | ".join(labels.drop_duplicates())


def unmatched_result(reason, available_ranges=""):
    """Represent an unresolved match without inventing a salary range."""
    return {
        "_salary_row_id": None, "matched_rank": None, "match_method": "unmatched",
        "unmatched_reason": reason, "available_range_titles": available_ranges,
    }


def select_equivalent_monthly_default(rows):
    """Choose a stable default only when usable monthly candidates have identical ranges."""
    usable = rows.loc[
        rows["Salary Per"].eq("M")
        & rows["Annual Factor"].gt(0)
        & rows["Min Salary"].notna()
        & rows["Max Salary"].ge(rows["Min Salary"])
    ]
    comparison_columns = ["_range_key", "Salary Per", "Annual Factor", "Min Salary", "Max Salary"]
    if usable.empty or len(usable[comparison_columns].drop_duplicates()) != 1:
        return rows.iloc[:0]
    # Class codes are four-digit CSU codes. Sorting avoids relying on CSV order.
    return usable.sort_values(["Class", "Effective Date"], ascending=[True, False]).head(1)


def select_unique_range(rows, method, available_ranges):
    """Select one row or the documented equivalent-monthly default."""
    if len(rows) > 1 and ASSIGN_EQUIVALENT_MONTHLY_RANGES:
        default = select_equivalent_monthly_default(rows)
        if not default.empty:
            rows = default
            method += " / assumed equivalent monthly range; lowest class code"
    if len(rows) != 1:
        return unmatched_result("Multiple schedule rows fit the rule", available_ranges)
    row = rows.iloc[0]
    return {
        "_salary_row_id": int(row["_salary_row_id"]),
        "matched_rank": row["_range_key"] or None, "match_method": method,
        "unmatched_reason": "", "available_range_titles": available_ranges,
    }

In [43]:
def match_lecturer_default(candidates, available_ranges):
    """Apply the configured lecturer default regardless of an explicit lecturer level."""
    selected = candidates.loc[candidates["_range_key"].isin(lecturer_range_keys())]
    if selected.empty:
        return unmatched_result("Lecturer default not available", available_ranges)
    method = "role default: Range " + LECTURER_DEFAULT_RANGE.upper() + " / AY and 12 month"
    return select_unique_range(selected, method, available_ranges)


def match_compatible_rank(candidates, extracted_rank, available_ranges):
    """Keep professor ranks distinct, allowing the reference's combined rank labels."""
    if extracted_rank is None:
        return unmatched_result("Role has multiple ranges; title does not identify a rank", available_ranges)
    compatible = candidates["_range_key"].map(
        lambda label: extracted_rank in COMBINED_RANK_LABELS.get(label, {label})
    )
    if compatible.any():
        return select_unique_range(
            candidates.loc[compatible], "role plus compatible title rank", available_ranges
        )
    return unmatched_result("Title rank unavailable under role; role retained", available_ranges)


def match_salary_range(schedule, groups_by_role, role, extracted_rank, title_rule):
    """Apply the existing rule order: title override, single row, lecturer, then rank."""
    if role in EXCLUDED_COMPARISON_ROLES:
        return unmatched_result(EXCLUDED_ROLE_REASON)
    candidates = select_role_candidates(schedule, groups_by_role, role)
    candidates, override_method = apply_title_override(schedule, candidates, title_rule)
    if candidates.empty:
        reason = f"Role classification not available in the {SALARY_SCHEDULE_YEAR} schedule"
        return unmatched_result(reason)
    available_ranges = describe_available_ranges(candidates)
    if override_method is not None:
        return select_unique_range(candidates, override_method, available_ranges)
    # ASSUMPTION: a single eligible row needs no explicit rank, even if its label is blank.
    if len(candidates) == 1:
        return select_unique_range(candidates, "single salary range for role", available_ranges)
    if role == "lecturer":
        return match_lecturer_default(candidates, available_ranges)
    return match_compatible_rank(candidates, extracted_rank, available_ranges)


def build_salary_matcher(schedule):
    """Cache repeated role/title combinations for this run's schedule only."""
    groups_by_role = {role: rows for role, rows in schedule.groupby("_role_key")}

    @lru_cache(maxsize=None)
    def match(role, extracted_rank, title_rule):
        return match_salary_range(schedule, groups_by_role, role, extracted_rank, title_rule)

    # A new cache is built on each run, so edited assumptions cannot reuse old matches.
    return match

In [44]:
import numpy as np


def infer_rank_from_other_years(csu):
    """Fill the rank of generic 'Instructional Faculty' rows from the same person's other years.

    Same person = same campus + name. The closest year whose title states a rank wins
    (ties -> earlier year). If equally close years disagree, the row stays unranked.
    `rank_source` records where each rank came from ("title" or "inferred from <year>").
    """
    csu = csu.copy()
    csu["rank_source"] = pd.Series(pd.NA, index=csu.index, dtype="string")
    csu.loc[csu["extracted_rank"].notna(), "rank_source"] = "title"
    if INSTRUCTIONAL_FACULTY_TITLE_POLICY != "infer_from_other_years":
        return csu
    generic = csu["title"].map(is_generic_instructional_faculty) & csu["extracted_rank"].isna()
    known = csu.loc[
        csu["extracted_rank"].isin(INFERABLE_RANKS) & ~csu["title"].map(is_generic_instructional_faculty),
        ["campus", "name", "year", "extracted_rank"],
    ].drop_duplicates()
    targets = csu.loc[generic, ["campus", "name", "year"]].reset_index().rename(columns={"index": "_row"})
    pairs = targets.merge(known, on=["campus", "name"], suffixes=("", "_known"))
    pairs["_gap"] = (pairs["year"] - pairs["year_known"]).abs()
    pairs = pairs.sort_values(["_row", "_gap", "year_known"])  # nearest year first, ties -> earlier
    nearest = pairs.groupby("_row").first()
    conflict = (
        pairs.merge(nearest[["_gap", "year_known"]], on=["_row", "_gap", "year_known"])
        .groupby("_row")["extracted_rank"].nunique().gt(1)
    )
    nearest = nearest.loc[~nearest.index.isin(conflict[conflict].index)]
    csu.loc[nearest.index, "extracted_rank"] = nearest["extracted_rank"].array
    csu.loc[nearest.index, "rank_source"] = "inferred from " + nearest["year_known"].astype(str).array
    return csu


def add_title_matching_fields(csu_data):
    """Preserve original columns and add the rank, adjusted role and explicit title-rule fields."""
    csu = csu_data.copy()
    csu["extracted_rank"] = csu["title"].map(extract_title_rank)
    csu = infer_rank_from_other_years(csu)
    csu["title_rule"] = [
        determine_title_rule(title, rank)
        for title, rank in zip(csu["title"], csu["extracted_rank"])
    ]
    # ASSUMPTION: a ladder_faculty payroll line whose title is a lecturer title is treated as a
    # lecturer. `role` itself is never overwritten; matching uses `role_adjusted`.
    lecturer_title = csu["extracted_rank"].fillna("").str.startswith("lecturer")
    csu["role_adjusted"] = np.where(
        csu["role"].eq("ladder_faculty") & lecturer_title, "lecturer", csu["role"]
    )
    return csu


def match_csu_rows(csu, matcher):
    """Return one selection per input row, preserving input order."""
    choices = [
        matcher(None if pd.isna(role) else role, None if pd.isna(rank) else rank, rule)
        for role, rank, rule in zip(csu["role_adjusted"], csu["extracted_rank"], csu["title_rule"])
    ]
    columns = ["_salary_row_id", "matched_rank", "match_method",
               "unmatched_reason", "available_range_titles"]
    selections = pd.DataFrame(choices, columns=columns)
    selections["_salary_row_id"] = pd.array(selections["_salary_row_id"], dtype="Int64")
    return selections


def join_salary_schedule(csu_data, schedule):
    """Attach the selected schedule row without duplicating or dropping personnel rows."""
    csu = add_title_matching_fields(csu_data)
    selections = match_csu_rows(csu, build_salary_matcher(schedule))
    for column in selections:
        csu[column] = selections[column].array
    public_schedule = schedule.drop(columns=["_role_key", "_class_key", "_range_key"])
    return csu.merge(
        public_schedule, on="_salary_row_id", how="left", validate="many_to_one",
        indicator="salary_match", suffixes=("", "_schedule"), sort=False,
    ).drop(columns="_salary_row_id")


def apply_final_range_labels(joined_data):
    """Add the comparison year and apply display labels without changing source data."""
    final = joined_data.copy()
    final["salary_schedule_year"] = SALARY_SCHEDULE_YEAR
    range_keys = final["Range Title"].map(normalize_title)
    for key, label in FINAL_RANGE_LABELS.items():
        final.loc[range_keys.eq(key), ["Range Title", "matched_rank"]] = label
    return final


def validate_final_csu_data(final, original, cutoff):
    """Check that original records survive unchanged and no future rates were joined."""
    assert len(final) == len(original)
    pd.testing.assert_frame_equal(
        final[original.columns].reset_index(drop=True), original.reset_index(drop=True)
    )
    matched_dates = final.loc[final["salary_match"].eq("both"), "Effective Date"]
    assert matched_dates.notna().all() and matched_dates.le(cutoff).all()

In [45]:
# After changing assumptions above, rerun from the assumptions cell through this cell.
# Inputs remain the golden CSU data and salary CSV loaded in your original cells.
schedule = build_matching_schedule(salary_schedule_df, SALARY_CUTOFF)
csu_salary_joined_df = join_salary_schedule(csu_data_df, schedule)
final_csu_df = apply_final_range_labels(csu_salary_joined_df)
validate_final_csu_data(final_csu_df, csu_data_df, SALARY_CUTOFF)
final_csu_df.head()

,year,campus,name,title,role,original_department,department,gross_pay,base_pay,overtime_pay,...,Step 25,Step 26,Step 27,Step 28,Step 29,Step 30,Step 31,Step 32,salary_match,salary_schedule_year
0,2013,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,Economics,"Business, Economics & Finance",148840,97454,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,2024
1,2014,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,Economics,"Business, Economics & Finance",164953,98207,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,2024
2,2015,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,Economics,"Business, Economics & Finance",155361,100861,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,2024
3,2016,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,Economics,"Business, Economics & Finance",146317,102539,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,2024
4,2016,CSU Bakersfield,Abbas Grammy,Professor Emeritus,supplemental_teaching_pay,Economics,"Business, Economics & Finance",0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2024


### Expected salary and comparison summaries

`salary_range_mean` is the arithmetic midpoint of `Min Salary` and `Max Salary`,
in the schedule's original pay unit. `salary_low`, `salary_high`, and `salary_expected`
are annualized using `Annual Factor` for monthly schedules only.

Assumptions: compare annual `gross_pay` (total pay) to those annualized amounts.
"Below range" means strictly below the minimum; "below expected" means strictly below
the midpoint, and includes below-range records. The midpoint is a comparison benchmark,
not a prediction of actual earnings. These comparisons use the 2024 schedule for all years,
without FTE adjustment, partial-year proration, or inflation adjustment.

Percentages below range/expected use **comparable records** in each group as the denominator:
matched records with nonmissing comparison pay and valid monthly bounds and annual factors.
Unmatched records and matched records without usable comparison data are shown separately.
A group with no comparable records has missing percentages, not zero percent.
Missing department/campus values remain included. Counts represent records, not unique people.

In [46]:
# Change this to another annual pay column only if that is the intended comparison.
PAY_COMPARISON_COLUMN = "gross_pay"


def add_salary_comparison_columns(data):
    """Add the raw range midpoint and compatible annual comparison amounts."""
    result = data.copy()
    minimum = parse_salary_amounts(result["Min Salary"])
    maximum = parse_salary_amounts(result["Max Salary"])
    factor = pd.to_numeric(result["Annual Factor"], errors="coerce")
    valid_range = (result["salary_match"].eq("both") & minimum.ge(0)
                   & maximum.ge(minimum)).fillna(False)
    result["salary_range_mean"] = ((minimum + maximum) / 2).where(valid_range)
    # Hourly/session schedules are not annualized without hours/session information.
    monthly = result["Salary Per"].astype("string").str.strip().eq("M").fillna(False).astype(bool)
    annualizable = (valid_range & monthly & factor.gt(0)).fillna(False)
    result["salary_low"] = (minimum * factor).where(annualizable)
    result["salary_high"] = (maximum * factor).where(annualizable)
    result["salary_expected"] = (result["salary_low"] + result["salary_high"]) / 2
    return result


def filter_summary_year(data):
    """Restrict a table to SUMMARY_YEAR (no change when SUMMARY_YEAR is None)."""
    return data if SUMMARY_YEAR is None else data.loc[data["year"].eq(SUMMARY_YEAR)]


def summarize_salary_comparisons(data, group_column):
    """Count records per group; excluded-role and zero-pay records are shown but never compared."""
    data = filter_summary_year(data)
    pay = parse_salary_amounts(data[PAY_COMPARISON_COLUMN])
    matched = data["salary_match"].eq("both")
    excluded = data["unmatched_reason"].eq(EXCLUDED_ROLE_REASON)
    pay_ok = pay.gt(MIN_PAY_FOR_COMPARISON).fillna(False)
    comparable = (matched & pay_ok & data["salary_low"].notna()
                  & data["salary_expected"].notna())
    flags = data[[group_column]].assign(
        excluded_role_records=excluded,
        unmatched_records=~matched & ~excluded,
        zero_pay_records=matched & pay.notna() & ~pay_ok,
        matched_not_comparable_records=matched & ~comparable & ~(pay.notna() & ~pay_ok),
        comparable_records=comparable,
        below_range_records=(comparable & pay.lt(data["salary_low"])).fillna(False),
        below_expected_records=(comparable & pay.lt(data["salary_expected"])).fillna(False),
    )
    summary = flags.groupby(group_column, dropna=False).agg(
        total_records=("unmatched_records", "size"),
        excluded_role_records=("excluded_role_records", "sum"),
        unmatched_records=("unmatched_records", "sum"),
        zero_pay_records=("zero_pay_records", "sum"),
        matched_not_comparable_records=("matched_not_comparable_records", "sum"),
        comparable_records=("comparable_records", "sum"),
        below_range_records=("below_range_records", "sum"),
        below_expected_records=("below_expected_records", "sum"),
    )
    denominator = summary["comparable_records"].replace(0, float("nan"))
    summary["below_range_percent"] = (100 * summary["below_range_records"] / denominator).round(2)
    summary["below_expected_percent"] = (100 * summary["below_expected_records"] / denominator).round(2)
    return summary


def summarize_record_counts(data, group_column):
    """Count all records in each group, including groups with a missing label."""
    summary = data.groupby(group_column, dropna=False).size().rename("records").to_frame()
    summary["percent_of_all_records"] = (100 * summary["records"] / len(data)).round(2)
    return summary


final_csu_df = add_salary_comparison_columns(final_csu_df)

In [47]:
# Department buckets are stored in department; campus is the school identifier.
department_salary_summary = summarize_salary_comparisons(final_csu_df, "department")
department_salary_summary

,total_records,excluded_role_records,unmatched_records,zero_pay_records,matched_not_comparable_records,comparable_records,below_range_records,below_expected_records,below_range_percent,below_expected_percent
department,,,,,,,,,,
"Agriculture, Food & Animal",566,143,11,60,0,352,131,293,37.22,83.24
Architecture & Environmental Design,447,98,22,37,0,290,133,260,45.86,89.66
"Business, Economics & Finance",2153,757,72,171,0,1153,262,514,22.72,44.58
Computer Science,985,307,48,85,0,545,154,338,28.26,62.02
Education & Public Administration,1668,498,69,154,0,947,408,794,43.08,83.84
Engineering,853,186,54,121,0,492,112,361,22.76,73.37
Humanities & Languages,3690,951,164,337,0,2238,820,1941,36.64,86.73
Medicine & Health Sciences,1366,412,37,94,0,823,337,687,40.95,83.48
Other,1030,283,265,83,0,399,238,317,59.65,79.45


In [48]:
school_salary_summary = summarize_salary_comparisons(final_csu_df, "campus")
school_salary_summary

,total_records,excluded_role_records,unmatched_records,zero_pay_records,matched_not_comparable_records,comparable_records,below_range_records,below_expected_records,below_range_percent,below_expected_percent
campus,,,,,,,,,,
CSU Bakersfield,437,120,0,12,0,305,99,225,32.46,73.77
CSU Channel Islands,578,229,21,24,0,304,124,225,40.79,74.01
CSU Long Beach,1405,379,113,121,0,792,377,652,47.6,82.32
CSU Los Angeles,583,178,314,11,0,80,43,59,53.75,73.75
CSU Northridge,838,310,0,8,0,520,114,436,21.92,83.85
CSU San Bernardino,1308,341,12,88,0,867,464,729,53.52,84.08
Cal Poly Humboldt,589,107,15,86,0,381,152,341,39.9,89.5
Cal Poly Pomona,974,327,1,30,0,616,218,472,35.39,76.62
Cal Poly San Luis Obispo,1002,57,3,289,0,653,104,512,15.93,78.41


In [49]:
records_by_year = summarize_record_counts(final_csu_df, "year")
records_by_year

,records,percent_of_all_records
year,,
2013,6414,4.12
2014,6987,4.49
2015,7652,4.92
2016,11847,7.62
2017,8684,5.58
2018,13630,8.77
2019,14818,9.53
2020,15343,9.87
2021,16086,10.34


In [50]:
records_by_school = summarize_record_counts(filter_summary_year(final_csu_df), "campus")
records_by_school

,records,percent_of_all_records
campus,,
CSU Bakersfield,437,2.33
CSU Channel Islands,578,3.08
CSU Long Beach,1405,7.48
CSU Los Angeles,583,3.10
CSU Northridge,838,4.46
CSU San Bernardino,1308,6.96
Cal Poly Humboldt,589,3.13
Cal Poly Pomona,974,5.18
Cal Poly San Luis Obispo,1002,5.33


### Person-year pay totals

A person can appear on several payroll lines in one year. These cells sum pay for each
year + campus + name + department so that one person-year is one row. This does not change
`final_csu_df`. There is no employee ID, so two different people with the same name, campus and
department in the same year would be combined. The summary cell follows `SUMMARY_YEAR` and
`MIN_PAY_FOR_COMPARISON` from the assumptions cell.


In [51]:
PERSON_YEAR_KEYS = ["year", "campus", "name", "department"]
PAY_COLUMNS = ["gross_pay", "base_pay", "overtime_pay", "other_pay"]

person_year_df = (
    csu_data_df.groupby(PERSON_YEAR_KEYS, dropna=False, as_index=False)
    .agg(**{"total_" + c: (c, "sum") for c in PAY_COLUMNS}, payroll_lines=("gross_pay", "size"))
)

# Which roles/titles were combined into each person-year (for inspection).
for column in ["role", "title"]:
    distinct = csu_data_df.dropna(subset=[column]).drop_duplicates(PERSON_YEAR_KEYS + [column])
    joined = (distinct.sort_values(column).groupby(PERSON_YEAR_KEYS, dropna=False)[column]
              .agg(" | ".join).rename(column + "s").reset_index())
    person_year_df = person_year_df.merge(joined, on=PERSON_YEAR_KEYS, how="left", validate="one_to_one")

assert person_year_df["total_gross_pay"].sum() == csu_data_df["gross_pay"].sum()  # nothing lost
assert person_year_df["payroll_lines"].sum() == len(csu_data_df)
person_year_df.head()

,year,campus,name,department,total_gross_pay,total_base_pay,total_overtime_pay,total_other_pay,payroll_lines,roles,titles
0,2013,CSU Bakersfield,Abbas Grammy,"Business, Economics & Finance",148840,97454,0,51386,1,ladder_faculty,Professor Emeritus
1,2013,CSU Bakersfield,Abigael Flachmann,Humanities & Languages,38033,33615,0,4418,1,lecturer,Lecturer in English
2,2013,CSU Bakersfield,Alemseghed Kebede,Social Sciences & Communication,77329,73680,0,3648,1,ladder_faculty,Professor
3,2013,CSU Bakersfield,Alexander Dzyubenko,Engineering,82383,82383,0,0,1,ladder_faculty,Professor Emeritus
4,2013,CSU Bakersfield,Allison Evans Fluckey,Social Sciences & Communication,57629,57105,0,524,1,lecturer,Lecturer in Psychology


In [52]:
py = filter_summary_year(person_year_df)
active = py.loc[py["total_gross_pay"].gt(MIN_PAY_FOR_COMPARISON)]

overall_person_year_summary = pd.Series({
    "person_years": len(py),
    "zero_total_pay": int((~py["total_gross_pay"].gt(MIN_PAY_FOR_COMPARISON)).sum()),
    "multi_line_person_years": int(py["payroll_lines"].gt(1).sum()),
    "mean_total_pay": active["total_gross_pay"].mean(),
    "median_total_pay": active["total_gross_pay"].median(),
    "p25_total_pay": active["total_gross_pay"].quantile(0.25),
    "p75_total_pay": active["total_gross_pay"].quantile(0.75),
}).round(0)

person_year_summary_by_campus = active.groupby("campus").agg(
    person_years=("total_gross_pay", "size"),
    multi_line_person_years=("payroll_lines", lambda s: int((s > 1).sum())),
    mean_total_pay=("total_gross_pay", "mean"),
    median_total_pay=("total_gross_pay", "median"),
).round(0)

display(overall_person_year_summary)
person_year_summary_by_campus

person_years                11701.0
zero_total_pay                344.0
multi_line_person_years      6047.0
mean_total_pay             101496.0
median_total_pay           105815.0
p25_total_pay               61645.0
p75_total_pay              135049.0
dtype: float64

,person_years,multi_line_person_years,mean_total_pay,median_total_pay
campus,,,,
CSU Bakersfield,292,135,112471.0,106095.0
CSU Channel Islands,328,212,89387.0,87633.0
CSU Long Beach,830,385,89566.0,77632.0
CSU Los Angeles,362,186,110200.0,120109.0
CSU Northridge,518,312,121884.0,123860.0
CSU San Bernardino,832,365,77801.0,69556.0
Cal Poly Humboldt,360,132,90578.0,88855.0
Cal Poly Pomona,605,327,98362.0,96190.0
Cal Poly San Luis Obispo,640,309,123589.0,125302.0


In [53]:
# Combine CSU (with salary-schedule columns) and UC into one long dataset.
# Outer-style union: every column from either side is kept; UC rows get NaN
# for the CSU-only salary schedule columns (the schedule only covers CSU).
SOURCE_COLUMN = "source_system"

csu_part = final_csu_df.copy()
uc_part = uc_data_df.copy()
assert SOURCE_COLUMN not in csu_part.columns and SOURCE_COLUMN not in uc_part.columns

csu_part[SOURCE_COLUMN] = "CSU"
uc_part[SOURCE_COLUMN] = "UC"

# merge indicator is categorical; object avoids category-vs-missing-column dtype surprises
if "salary_match" in csu_part.columns:
    csu_part["salary_match"] = csu_part["salary_match"].astype("object")

# Column order: shared payroll columns first (same order for both), then
# source_system, then CSU-only salary columns.
shared_cols = [c for c in csu_data_df.columns if c in uc_data_df.columns]
extra_cols = [c for c in csu_part.columns if c not in shared_cols and c != SOURCE_COLUMN]
ordered_cols = shared_cols + [SOURCE_COLUMN] + extra_cols

combined_df = pd.concat(
    [csu_part[ordered_cols], uc_part[shared_cols + [SOURCE_COLUMN]]],
    axis=0, join="outer", ignore_index=True, sort=False,
)

# Nothing dropped: every column from both inputs is present, and every row survives.
assert set(combined_df.columns) == set(csu_part.columns) | set(uc_part.columns)
assert len(combined_df) == len(csu_part) + len(uc_part)
assert combined_df[SOURCE_COLUMN].value_counts().to_dict() == {"UC": len(uc_part), "CSU": len(csu_part)}

combined_df.groupby(SOURCE_COLUMN).size()

source_system
CSU    155500
UC     399975
dtype: int64

In [56]:
combined_df.head()

,year,campus,name,title,role,original_department,department,gross_pay,base_pay,overtime_pay,...,Step 29,Step 30,Step 31,Step 32,salary_match,salary_schedule_year,salary_range_mean,salary_low,salary_high,salary_expected
0,2013,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,Economics,"Business, Economics & Finance",148840,97454,0,...,NaN,NaN,NaN,NaN,both,2024.0,13041.5,103116.0,209880.0,156498.0
1,2014,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,Economics,"Business, Economics & Finance",164953,98207,0,...,NaN,NaN,NaN,NaN,both,2024.0,13041.5,103116.0,209880.0,156498.0
2,2015,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,Economics,"Business, Economics & Finance",155361,100861,0,...,NaN,NaN,NaN,NaN,both,2024.0,13041.5,103116.0,209880.0,156498.0
3,2016,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,Economics,"Business, Economics & Finance",146317,102539,0,...,NaN,NaN,NaN,NaN,both,2024.0,13041.5,103116.0,209880.0,156498.0
4,2016,CSU Bakersfield,Abbas Grammy,Professor Emeritus,supplemental_teaching_pay,Economics,"Business, Economics & Finance",0,0,0,...,NaN,NaN,NaN,NaN,left_only,2024.0,<NA>,<NA>,<NA>,<NA>


In [55]:
from pathlib import Path

output_path = Path("./golden-datasets/csu_uc_salary_schedule_2013-2024.parquet")
combined_df.to_parquet(output_path, index=False)
print(f"Saved {len(combined_df):,} records to {output_path.resolve()}")

Saved 555,475 records to C:\Users\ginny\Classes-Other\DataEng\CSC4693\Data-Folding\golden-datasets\csu_uc_salary_schedule_2013-2024.parquet
